In [1]:
import numpy as np
import pandas as pd

In [2]:
prices = pd.read_csv('clean/prices.csv', parse_dates=['timestamp'])
load = pd.read_csv('clean/load.csv', parse_dates=['timestamp'])
ren = pd.read_csv('clean/renewables_weather.csv', parse_dates=['timestamp'])

In [3]:
prices

,timestamp,price_eur_mwh,daily_avg_price,dst_adjusted
0,2021-01-01 00:00:00,44.98,47.91,False
1,2021-01-01 01:00:00,40.82,47.91,False
2,2021-01-01 02:00:00,38.19,47.91,False
3,2021-01-01 03:00:00,33.17,47.91,False
4,2021-01-01 04:00:00,31.69,47.91,False
...,...,...,...,...
26275,2023-12-31 19:00:00,105.78,84.32,False
26276,2023-12-31 20:00:00,106.10,84.32,False
26277,2023-12-31 21:00:00,99.21,84.32,False
26278,2023-12-31 22:00:00,87.20,84.32,False


In [4]:
prices = prices.rename(columns={'price_eur_mwh': 'price'})
prices.loc[prices['price'] == -9999, 'price'] = np.nan

prices['hour'] = prices['timestamp'].dt.hour
prices['date'] = prices['timestamp'].dt.normalize()

prices['p_lag1'] = prices.groupby('hour')['price'].shift(1)
prices['p_lag7'] = prices.groupby('hour')['price'].shift(7)

prices['daily_p_lag1'] = prices.groupby('hour')['daily_avg_price'].shift(1)
prices['lvl28'] = prices.groupby('hour')['daily_p_lag1'].rolling(28).median().reset_index(level=0)['daily_p_lag1']

In [5]:
df = pd.merge(prices, load[['timestamp', 'load_forecast_mw']], on='timestamp', how='left')
df = df.rename(columns={'load_forecast_mw': 'load_fc'})
df

,timestamp,price,daily_avg_price,dst_adjusted,hour,date,p_lag1,p_lag7,daily_p_lag1,lvl28,load_fc
0,2021-01-01 00:00:00,44.98,47.91,False,0,2021-01-01,NaN,NaN,NaN,NaN,40341.0
1,2021-01-01 01:00:00,40.82,47.91,False,1,2021-01-01,NaN,NaN,NaN,NaN,39685.0
2,2021-01-01 02:00:00,38.19,47.91,False,2,2021-01-01,NaN,NaN,NaN,NaN,38683.0
3,2021-01-01 03:00:00,33.17,47.91,False,3,2021-01-01,NaN,NaN,NaN,NaN,38875.0
4,2021-01-01 04:00:00,31.69,47.91,False,4,2021-01-01,NaN,NaN,NaN,NaN,39599.0
...,...,...,...,...,...,...,...,...,...,...,...
26275,2023-12-31 19:00:00,105.78,84.32,False,19,2023-12-31,102.28,83.97,101.26,153.44,47427.0
26276,2023-12-31 20:00:00,106.10,84.32,False,20,2023-12-31,91.23,82.85,101.26,153.44,46626.0
26277,2023-12-31 21:00:00,99.21,84.32,False,21,2023-12-31,71.92,84.37,101.26,153.44,44438.0
26278,2023-12-31 22:00:00,87.20,84.32,False,22,2023-12-31,73.74,67.48,101.26,153.44,41007.0


In [6]:
df = pd.merge(df, ren[['timestamp', 'wind_forecast_mw', 'solar_forecast_mw']], on='timestamp', how='left')
df = df.rename(columns={'wind_forecast_mw': 'wind_fc', 'solar_forecast_mw': 'solar_fc'})
df['resid_fc'] = df['load_fc'] - df['wind_fc'] - df['solar_fc']
df

,timestamp,price,daily_avg_price,dst_adjusted,hour,date,p_lag1,p_lag7,daily_p_lag1,lvl28,load_fc,wind_fc,solar_fc,resid_fc
0,2021-01-01 00:00:00,44.98,47.91,False,0,2021-01-01,NaN,NaN,NaN,NaN,40341.0,9799.0,0.0,30542.0
1,2021-01-01 01:00:00,40.82,47.91,False,1,2021-01-01,NaN,NaN,NaN,NaN,39685.0,9370.0,0.0,30315.0
2,2021-01-01 02:00:00,38.19,47.91,False,2,2021-01-01,NaN,NaN,NaN,NaN,38683.0,10324.0,0.0,28359.0
3,2021-01-01 03:00:00,33.17,47.91,False,3,2021-01-01,NaN,NaN,NaN,NaN,38875.0,9784.0,0.0,29091.0
4,2021-01-01 04:00:00,31.69,47.91,False,4,2021-01-01,NaN,NaN,NaN,NaN,39599.0,10656.0,0.0,28943.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26275,2023-12-31 19:00:00,105.78,84.32,False,19,2023-12-31,102.28,83.97,101.26,153.44,47427.0,2241.0,0.0,45186.0
26276,2023-12-31 20:00:00,106.10,84.32,False,20,2023-12-31,91.23,82.85,101.26,153.44,46626.0,2274.0,0.0,44352.0
26277,2023-12-31 21:00:00,99.21,84.32,False,21,2023-12-31,71.92,84.37,101.26,153.44,44438.0,1829.0,0.0,42609.0
26278,2023-12-31 22:00:00,87.20,84.32,False,22,2023-12-31,73.74,67.48,101.26,153.44,41007.0,1743.0,0.0,39264.0


# Task 2

In [2]:
bars = pd.read_csv('mock3_data/bars.csv', parse_dates=['ts'])
vendor = pd.read_csv('mock3_data/vendor_signal.csv', parse_dates=['published_at'])

In [3]:
bars['date'] = bars['ts'].dt.normalize()
fwd_mid_5 = bars[['ts', 'instrument', 'mid_close']]
fwd_mid_5['ts'] = bars['ts'] - pd.Timedelta(minutes=5)
fwd_mid_5 = fwd_mid_5.rename(columns={'mid_close': 'fwd_mid_5'})
fwd_mid_5

C:\Users\20122\AppData\Local\Temp\ipykernel_20752\754378722.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fwd_mid_5['ts'] = bars['ts'] - pd.Timedelta(minutes=5)


,ts,instrument,fwd_mid_5
0,2024-01-02 09:26:00,HRN,1998.300
1,2024-01-02 09:26:00,KST,4997.625
2,2024-01-02 09:26:00,PLV,798.850
3,2024-01-02 09:27:00,HRN,1999.000
4,2024-01-02 09:27:00,KST,4997.625
...,...,...,...
145743,2024-06-28 15:54:00,KST,4042.125
145744,2024-06-28 15:54:00,PLV,691.625
145745,2024-06-28 15:55:00,HRN,1529.600
145746,2024-06-28 15:55:00,KST,4042.125


In [ ]:
bars = pd.merge(bars, fwd_mid_5, on=['ts', 'instrument'], how='left')
bars

,ts,instrument,mid_close,spread_close,last_price,buy_volume,sell_volume,n_trades,date,fwd_mid_5
0,2024-01-02 09:31:00,HRN,1998.300,0.20,1998.20,80,474,268,2024-01-02,1999.900
1,2024-01-02 09:31:00,KST,4997.625,0.25,4997.75,1269,978,585,2024-01-02,4992.125
2,2024-01-02 09:31:00,PLV,798.850,0.10,798.80,28,570,316,2024-01-02,797.150
3,2024-01-02 09:32:00,HRN,1999.000,0.20,1998.90,191,319,241,2024-01-02,2003.500
4,2024-01-02 09:32:00,KST,4997.625,0.25,4997.75,265,1161,334,2024-01-02,4997.875
...,...,...,...,...,...,...,...,...,...,...
145858,2024-06-28 15:59:00,KST,4042.125,0.25,4042.25,5570,12,1476,2024-06-28,NaN
145859,2024-06-28 15:59:00,PLV,691.625,0.15,691.70,871,4,410,2024-06-28,NaN
145860,2024-06-28 16:00:00,HRN,1529.600,0.20,1529.70,329,168,256,2024-06-28,NaN
145861,2024-06-28 16:00:00,KST,4042.125,0.25,4042.00,241,698,251,2024-06-28,NaN


In [5]:
bars['fwd_ret_5'] = np.log(bars['fwd_mid_5']) - np.log(bars['mid_close'])

In [7]:
vendor['date'] = vendor['published_at'].dt.normalize()
vendor

,published_at,instrument,signal,date
0,2024-01-02 09:35:00,HRN,-0.1991,2024-01-02
1,2024-01-02 09:35:00,KST,-1.8237,2024-01-02
2,2024-01-02 09:35:00,PLV,0.2963,2024-01-02
3,2024-01-02 09:40:00,HRN,-0.6262,2024-01-02
4,2024-01-02 09:40:00,KST,1.2475,2024-01-02
...,...,...,...,...
29133,2024-06-28 15:55:00,KST,0.5844,2024-06-28
29134,2024-06-28 15:55:00,PLV,-1.1904,2024-06-28
29135,2024-06-28 16:00:00,HRN,1.8414,2024-06-28
29136,2024-06-28 16:00:00,KST,-0.6762,2024-06-28


In [ ]:
pd.merge_asof(bars, vendor, left_on='ts', right_on='published_at', by=['instrument', 'date'])

,ts,instrument,mid_close,spread_close,last_price,buy_volume,sell_volume,n_trades,date,fwd_mid_5,fwd_ret_5,published_at,signal
0,2024-01-02 09:31:00,HRN,1998.300,0.20,1998.20,80,474,268,2024-01-02,1999.900,0.000800,NaT,NaN
1,2024-01-02 09:31:00,KST,4997.625,0.25,4997.75,1269,978,585,2024-01-02,4992.125,-0.001101,NaT,NaN
2,2024-01-02 09:31:00,PLV,798.850,0.10,798.80,28,570,316,2024-01-02,797.150,-0.002130,NaT,NaN
3,2024-01-02 09:32:00,HRN,1999.000,0.20,1998.90,191,319,241,2024-01-02,2003.500,0.002249,NaT,NaN
4,2024-01-02 09:32:00,KST,4997.625,0.25,4997.75,265,1161,334,2024-01-02,4997.875,0.000050,NaT,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
145858,2024-06-28 15:59:00,KST,4042.125,0.25,4042.25,5570,12,1476,2024-06-28,NaN,NaN,2024-06-28 15:55:00,0.5844
145859,2024-06-28 15:59:00,PLV,691.625,0.15,691.70,871,4,410,2024-06-28,NaN,NaN,2024-06-28 15:55:00,-1.1904
145860,2024-06-28 16:00:00,HRN,1529.600,0.20,1529.70,329,168,256,2024-06-28,NaN,NaN,2024-06-28 16:00:00,1.8414
145861,2024-06-28 16:00:00,KST,4042.125,0.25,4042.00,241,698,251,2024-06-28,NaN,NaN,2024-06-28 16:00:00,-0.6762


# Task 3

In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

def monthly_splits(df, start, end, train_months):
    for m in pd.date_range(start, end, freq='MS'):
        train = (df['date'] >= m - pd.DateOffset(months=train_months)) & (df['date'] < m)
        test = (df['date'] >= m) & (df['date'] < m + pd.DateOffset(months=1))
        yield (m, train, test)

def walk_forward(df, features, make_model, target='price', start='2022-01-01', end='2023-12-01', train_months=3):
    out = []
    for m, tr, te in monthly_splits(df, start, end, train_months):
        train_fold_df = df.loc[tr, features+[target]].dropna()
        model = make_model().fit(train_fold_df[features], train_fold_df[target])
        test_fold_df = df.loc[te, features+['date', 'hour', target]].dropna()
        pred = test_fold_df[['date', 'hour', 'price']].copy()
        pred['pred_price'] = model.predict(test_fold_df[features])
        out.append(pred)
    return pd.concat(out)

features = ['p_lag1', 'p_lag7', 'lvl28', 'resid_fc']
make_model = lambda: LinearRegression()
pred = walk_forward(df, features, make_model)
print(mean_absolute_error(pred['price'], pred['pred_price']))

46.98876553945653


In [9]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Lasso
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import ColumnTransformer

num_cols = ['p_lag1', 'p_lag7', 'lvl28', 'resid_fc']
cat_cols = ['hour']
features = num_cols + cat_cols

def make_lasso():
    pre = ColumnTransformer([
        ('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ])
    return make_pipeline(pre, Lasso(alpha=0.1, max_iter=20000))

make_tree = lambda: HistGradientBoostingRegressor(random_state=0)

lasso_pred = walk_forward(df, features, make_lasso)
print(mean_absolute_error(lasso_pred['price'], lasso_pred['pred_price']))

tree_pred = walk_forward(df, features, make_tree)
print(mean_absolute_error(tree_pred['price'], tree_pred['pred_price']))

ValueError: The feature names should match those that were passed during fit.
Feature names must be in the same order as they were in fit.
